# Assigning coords as dimension coordinates

The `gridmarthe` package use "horizontally reduced" grids, which means that
the spatial dimension is flattened. This is a common practice in geophysical
modeling (meteorology, oceanography, land surface modelling, etc.), where the
grid is often represented as a 1D array of points.

Grids are often 2D in space (or 3D), but here represented as 1D vector of size
`Nx*Ny` (or `Nx*Ny*Nz`). Each grid point is associated to an index and a set of
coordinates (and `dx`, `dy`).

This is particularly useful and efficient for computing on grids. But as a
consequence, it is not straightforward to work with such data, especially when
plotting.

This notebooks explore how to move coordinates from variables (1D associated to
indices) to coordinates (2D associated to dimensions).

In [2]:
# import package
import gridmarthe as gm

In [3]:
# reading data and storing into Dataset object (with xarray)
ds = gm.load_marthe_grid(
    './data/chasim_hallue.out', 'CHARGE',
    fpastp='./data/hallue.pastp',
    drop_nan=True
)

In [4]:
# display values and attributes
display(ds)

<xarray.Dataset> Size: 2MB
Dimensions:  (time: 205, zone: 927)
Coordinates:
  * time     (time) datetime64[ns] 2kB 1995-07-31 1995-08-01 ... 2012-07-01
  * zone     (zone) int32 4kB 255 256 257 258 259 ... 2722 2723 2724 2725 2726
Data variables:
    charge   (time, zone) float64 2MB 100.0 100.6 101.1 ... 27.0 26.0 26.35
    x        (zone) float32 4kB 617.8 618.2 618.8 619.2 ... 606.8 607.2 607.8
    y        (zone) float32 4kB 2.567e+03 2.567e+03 ... 2.543e+03 2.543e+03
    dx       (zone) float32 4kB 0.5 0.5 0.5 0.5 0.5 0.5 ... 0.5 0.5 0.5 0.5 0.5
    dy       (zone) float32 4kB 0.5 0.5 0.5 0.5 0.5 0.5 ... 0.5 0.5 0.5 0.5 0.5
    izone    (zone) int32 4kB 1 2 3 4 5 6 7 8 ... 921 922 923 924 925 926 927
Attributes: (12/17)
    conventions:          CF-1.10
    title:                Modélisation du bassin de la SOMME Nappe_Libre
    marthe_grid_version:  9.0
    original_dimensions:  x,y,z [grids]: 53 54 1
    crs:                  {'crs_wkt': 'PROJCRS["NTF (Paris) / Lambert zone II...
    lon_resolution:       0.5
    ...                   ...
    period:               1995-2012
    frequency:            30 day(s)
    creation_date:        Created on 2026-07-16T11:04:28Z UTC
    comment:              Hydrogeological model created with MARTHE code (Thi...
    domain:               FR-France
    institution:          BRGM, French Geological Survey, Orléans, France

We have a time index `time` (timesteps).

The spatial index is called `zone` and coordinates variables are:

- `x` (x-coordinate of centre cell),
- `y` (y-coordinate of centre cell),
- `z` (indice of layer or real `z`) if present,
- `dx` (cell width),
- `dy` (cell height)
- `izone` (same as zone index, but reset to 1 and incremental only on active cells).

## Switch to 2 or 3D grid

To move x,y,z coordinates from variables (depending on zone index) to
array dimensions, and then reshape data from 1D (or 2D with time) to
3 (x,y, time) or 4D (x,y,z,time), use: `assign_coords`.

In [5]:
# add coordinates to transform the 2D Dataset into 3D
ds_3d = gm.assign_coords(ds)
display(ds_3d)  # check new coordinates

<xarray.Dataset> Size: 3MB
Dimensions:  (y: 48, x: 44, time: 205)
Coordinates:
  * y        (y) float32 192B 2.567e+03 2.566e+03 ... 2.544e+03 2.543e+03
  * x        (x) float32 176B 599.2 599.8 600.2 600.8 ... 619.8 620.2 620.8
  * time     (time) datetime64[ns] 2kB 1995-07-31 1995-08-01 ... 2012-07-01
Data variables:
    charge   (time, y, x) float64 3MB nan nan nan nan nan ... nan nan nan nan
    dx       (y, x) float32 8kB nan nan nan nan nan nan ... nan nan nan nan nan
    dy       (y, x) float32 8kB nan nan nan nan nan nan ... nan nan nan nan nan
    izone    (y, x) float64 17kB nan nan nan nan nan nan ... nan nan nan nan nan
Attributes: (12/17)
    conventions:          CF-1.10
    title:                Modélisation du bassin de la SOMME Nappe_Libre
    marthe_grid_version:  9.0
    original_dimensions:  x,y,z [grids]: 53 54 1
    crs:                  {'crs_wkt': 'PROJCRS["NTF (Paris) / Lambert zone II...
    lon_resolution:       0.5
    ...                   ...
    period:               1995-2012
    frequency:            30 day(s)
    creation_date:        Created on 2026-07-16T11:04:28Z UTC
    comment:              Hydrogeological model created with MARTHE code (Thi...
    domain:               FR-France
    institution:          BRGM, French Geological Survey, Orléans, France

Data are now 3D for the `charge` (head) variable: `y` and `x` are now coordinates
dimensions, with `time`.

This dataset can now be plot easily as show in the previous notebook.

## Return to 1D grid

It can still be useful to return to a 1D grid. To do so, use `stack_coords` function:

In [7]:
ds1d = gm.stack_coords(ds_3d, dropna=True)
display(ds1d)

<xarray.Dataset> Size: 2MB
Dimensions:  (time: 205, zone: 927)
Coordinates:
  * time     (time) datetime64[ns] 2kB 1995-07-31 1995-08-01 ... 2012-07-01
  * zone     (zone) int64 7kB 38 39 40 41 42 43 ... 2082 2083 2084 2085 2086
Data variables:
    charge   (time, zone) float64 2MB 100.0 100.6 101.1 ... 27.0 26.0 26.35
    dx       (zone) float32 4kB 0.5 0.5 0.5 0.5 0.5 0.5 ... 0.5 0.5 0.5 0.5 0.5
    dy       (zone) float32 4kB 0.5 0.5 0.5 0.5 0.5 0.5 ... 0.5 0.5 0.5 0.5 0.5
    izone    (zone) float64 7kB 1.0 2.0 3.0 4.0 5.0 ... 924.0 925.0 926.0 927.0
    y        (zone) float32 4kB 2.567e+03 2.567e+03 ... 2.543e+03 2.543e+03
    x        (zone) float32 4kB 617.8 618.2 618.8 619.2 ... 606.8 607.2 607.8
Attributes: (12/17)
    conventions:          CF-1.10
    title:                Modélisation du bassin de la SOMME Nappe_Libre
    marthe_grid_version:  9.0
    original_dimensions:  x,y,z [grids]: 53 54 1
    crs:                  {'crs_wkt': 'PROJCRS["NTF (Paris) / Lambert zone II...
    lon_resolution:       0.5
    ...                   ...
    period:               1995-2012
    frequency:            30 day(s)
    creation_date:        Created on 2026-07-16T11:04:28Z UTC
    comment:              Hydrogeological model created with MARTHE code (Thi...
    domain:               FR-France
    institution:          BRGM, French Geological Survey, Orléans, France

**Warning**: Mind that in the previous operation (`stack_coords`) the `zone`
index is not consistent anymore. New index are generated.